# 10. Revisión Clínica Externa (IPS)

**Objetivo:** consolidar material clínicamente interpretable para revisión clínica externa y preparación de casos reutilizables en la futura etapa de xAI.
**Entradas:** artefactos vigentes de `dev`, outputs de preprocesamiento, cierres de modelo, predicciones comparables y error analysis alineado.
**Salidas:** `data/outputs/material_validacion_ips_<timestamp>/`, `data/outputs/dossier_ips_curado_<timestamp>/` y `data/outputs/ips_cierre_final_<timestamp>/` con material clínico, dossier curado y paquete final de revisión externa.
**Notebook anterior:** `notebooks/analysis/09_analisis_errores_hibrido.ipynb`.
**Notebook siguiente:** ninguno (fase secundaria de revisión clínica externa).

> **Alcance actual:** tarea binaria `ansiedad` vs `depresion`; no hay grupo de control ni clase explícita de comorbilidad en esta fase; `test` y xAI siguen pendientes.


## Técnicas, herramientas y librerías de esta etapa

- **Técnica principal:** consolidación clínica-documental de artefactos ya cerrados en `dev`.
- **Herramientas/librerías:** el notebook usa como backend `scripts/export/generar_material_validacion_ips.py`, `scripts/export/curar_dossier_ips.py` y `scripts/export/cerrar_fase_ips.py`. Para visualización usa `pandas` e `IPython.display`.
- **Por qué es adecuada aquí:** la generación de paquetes y exportables repetibles es más robusta en script que en celdas manuales. El notebook queda como capa legible para revisión clínica externa, pero además orquesta los tres pasos para dejar la fase IPS autocontenida.
- **Limitación:** no reabre el entrenamiento, no usa `test` y no reemplaza la validación clínica externa final.
- **Alternativa menos recomendable para esta fase:** notebook monolítico con toda la lógica de exportación; más cómodo para exploración, peor para trazabilidad.


## Resumen ejecutivo del problema

Este notebook traduce el estado actual del pipeline en un paquete útil para psiquiatras y para escritura metodológica.
No cambia la tarea experimental, no reabre la selección del modelo y no usa `test`.

Su propósito es responder de forma explícita:

1. cómo se preprocesó y filtró el corpus;
2. con cuántas notas/pacientes se está modelando realmente;
3. cómo quedó el balance/desbalance y cómo se trató;
4. qué parece captar cada enfoque (`TF-IDF`, el mejor transformer standalone vigente y el híbrido final);
5. qué errores conviene revisar con IPS;
6. qué vacíos siguen dependiendo de validación experta.


In [ ]:

import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for cand in [start, *start.parents]:
        if (cand / 'data').exists() and (cand / 'scripts').exists():
            return cand
    raise RuntimeError('No se pudo resolver la raíz del repositorio.')

ROOT = find_repo_root(Path.cwd().resolve())
sys.path.insert(0, str(ROOT / 'scripts' / 'export'))

from generar_material_validacion_ips import generar_material  # noqa: E402

RUN_TAG = os.getenv('IPS_OUTPUT_TAG', '').strip() or pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
OUT_DIR = ROOT / 'data' / 'outputs' / f'material_validacion_ips_{RUN_TAG}'
manifest = generar_material(OUT_DIR, verbose=False)

cmd_curar = [
    sys.executable,
    str(ROOT / 'scripts' / 'export' / 'curar_dossier_ips.py'),
    '--source-dir', str(OUT_DIR),
    '--output-tag', RUN_TAG,
]
proc_curar = subprocess.run(cmd_curar, cwd=ROOT, text=True, capture_output=True)
if proc_curar.returncode != 0:
    print(proc_curar.stdout)
    print(proc_curar.stderr)
    raise RuntimeError('Falló la curación del dossier IPS.')

cmd_cerrar = [
    sys.executable,
    str(ROOT / 'scripts' / 'export' / 'cerrar_fase_ips.py'),
    '--output-tag', RUN_TAG,
]
proc_cerrar = subprocess.run(cmd_cerrar, cwd=ROOT, text=True, capture_output=True)
if proc_cerrar.returncode != 0:
    print(proc_cerrar.stdout)
    print(proc_cerrar.stderr)
    raise RuntimeError('Falló el cierre final de la fase IPS.')

latest_dossier = json.loads((ROOT / 'data' / 'outputs' / 'dossier_ips_curado_latest.json').read_text(encoding='utf-8'))
latest_ips = json.loads((ROOT / 'data' / 'outputs' / 'ips_cierre_final_latest.json').read_text(encoding='utf-8'))

{
    'material_manifest': manifest,
    'dossier_output_dir': latest_dossier.get('output_dir'),
    'ips_cierre_output_dir': latest_ips.get('output_dir'),
}


## 1. Resumen del preprocesamiento y filtrado

## 1b. Qué entendemos por señal clínica útil

Esta sección no redefine el pipeline. Solo vuelve explícita, para lectura clínica y metodológica, la política operativa centralizada en `utils_shared.keep_entity` y reutilizada por el cierre IPS final.

In [ ]:
latest_ips_ptr = ROOT / 'data' / 'outputs' / 'ips_cierre_final_latest.json'
ips_cierre_dir = None
if latest_ips_ptr.exists():
    latest_ips_payload = json.loads(latest_ips_ptr.read_text(encoding='utf-8'))
    ips_cierre_dir = Path(latest_ips_payload['output_dir'])
else:
    candidatos_ips = sorted((ROOT / 'data' / 'outputs').glob('ips_cierre_final_*'))
    ips_cierre_dir = candidatos_ips[-1] if candidatos_ips else None

if ips_cierre_dir is None or not ips_cierre_dir.exists():
    raise FileNotFoundError('No se encontró un paquete `ips_cierre_final_*` para mostrar la definición de señal clínica útil.')

display(Markdown((ips_cierre_dir / 'senal_clinica_util.md').read_text(encoding='utf-8')))

In [ ]:
display(pd.read_csv(OUT_DIR / 'material_ips_preprocesamiento_resumen.csv'))
display(Markdown((OUT_DIR / 'material_ips_preprocesamiento_resumen.md').read_text(encoding='utf-8')))


## 2. Tamaño final del dataset y balance/desbalance

## 2b. Contraste metodológico: baseline crudo vs filtrado

Este contraste no reemplaza las líneas base oficiales. Se usa para explicar por qué el denoising fue necesario y por qué el problema no debe describirse como si el universo crudo y el universo filtrado fueran equivalentes.

In [ ]:
display(Markdown((ips_cierre_dir / 'baseline_crudo_vs_filtrado.md').read_text(encoding='utf-8')))

In [ ]:
display(pd.read_csv(OUT_DIR / 'material_ips_balance_dataset.csv'))
display(Markdown((OUT_DIR / 'material_ips_balance_dataset.md').read_text(encoding='utf-8')))


## 3. Qué se hizo y qué no se hizo todavía

In [ ]:
display(Markdown((OUT_DIR / 'material_ips_hecho_vs_pendiente.md').read_text(encoding='utf-8')))


## 4. Patrones finales por clase

In [ ]:
display(Markdown((OUT_DIR / 'material_ips_patrones_ansiedad.md').read_text(encoding='utf-8')))
display(Markdown((OUT_DIR / 'material_ips_patrones_depresion.md').read_text(encoding='utf-8')))


## 5. Qué identifica cada modelo y cuánto se parecen

In [ ]:
display(pd.read_csv(OUT_DIR / 'comparacion_aportes_modelos.csv'))
display(Markdown((OUT_DIR / 'comparacion_aportes_modelos.md').read_text(encoding='utf-8')))


## 6. Errores del modelo para IPS

In [ ]:
errores_ips = pd.read_csv(OUT_DIR / 'material_ips_errores_modelo.csv')
display(errores_ips.head(20))
display(Markdown((OUT_DIR / 'material_ips_errores_modelo.md').read_text(encoding='utf-8')))


## 7. Preguntas estructuradas para IPS

In [ ]:
display(Markdown((OUT_DIR / 'preguntas_sugeridas_ips.md').read_text(encoding='utf-8')))


## 8. Justificación metodológica y apoyo de literatura

## 8b. Trazabilidad de scripts backend

Este notebook es una capa de lectura. La generación reproducible de artefactos se delega a scripts para evitar curación manual dentro del notebook:

- `scripts/export/generar_material_validacion_ips.py`: arma el paquete base de revisión clínica externa.
- `scripts/export/curar_dossier_ips.py`: genera el dossier clínico curado y el set de casos reutilizables para futura xAI.
- `scripts/export/cerrar_fase_ips.py`: consolida auditoría del dataset, contraste crudo vs filtrado y material metodológico complementario.

Esta separación es la opción adecuada en esta fase porque mejora trazabilidad y evita que el notebook quede como única fuente de verdad.


In [ ]:
display(Markdown((OUT_DIR / 'justificacion_metodologica_y_clinica.md').read_text(encoding='utf-8')))
display(Markdown((OUT_DIR / 'preguntas_bibliografia_validacion_clinica.md').read_text(encoding='utf-8')))
